In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

In [10]:
data_dir = r"C:\coffee-leaf-disease\dataset"

categories = ['healthy', 'miner', 'phoma', 'rust']

data = []
labels = []

In [11]:
def get_features(img):
    try:
        
        img = cv2.resize(img, (128, 128))

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        gray_flat = gray.flatten()

        hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
        hist = hist.flatten()

        lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
        lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10))
        lbp_hist = lbp_hist / lbp_hist.sum()
    
        return np.concatenate((gray_flat, hist, lbp_hist))

        return features

    except:
        return None

In [12]:
for label, folder in enumerate(categories):
    path = os.path.join(data_dir, folder)

    for file in os.listdir(path):
        img_path = os.path.join(path, file)

        img = cv2.imread(img_path)
        if img is None:
            continue

        features = get_features(img)

        if features is None:
            continue
        


        data.append(features)
        labels.append(label)

x = np.array(data)
y = np.array(labels)

print("Dataset shape:", x.shape)

Dataset shape: (0,)


In [27]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

In [28]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [ ]:
svm = SVC(
    kernel='linear',    
    C=5,
    probability=True,
    class_weight='balanced'
)

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss'
)

model = VotingClassifier(
    estimators=[('svm', svm), ('xgb', xgb)],
    voting='soft'
)

model.fit(x_train, y_train)

In [ ]:
pred = model.predict(x_test)

acc = accuracy_score(y_test, pred)

print("Accuracy:", acc)